# Entrenamiento y comparación (issue #10)
La comparación vive acá, en el notebook, con código sklearn propio: el backend solo sirve `arbol` ya ajustado y no compara. La preparación sí se reutiliza del backend (`ml_preparation`, semilla 0). Metodología: CV estratificada solo en train (5 folds), grilla chica, contraste en validación, test un solo uso.

In [ ]:
import sys
from pathlib import Path

CANDIDATOS = [Path.cwd() / "backend" / "src", Path.cwd().parent / "backend" / "src"]
SRC = next(c for c in CANDIDATOS if (c / "collection").is_dir())
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import collection
print("src:", SRC.exists(), "schema:", collection.SCHEMA_VERSION)


In [ ]:
ROOTS = [Path.cwd(), Path.cwd().parent]
DATASET = next(r / "dataset" / "salida" / "tramos.csv" for r in ROOTS if (r / "dataset" / "salida" / "tramos.csv").is_file())
print("dataset:", DATASET)


In [ ]:
from collection.adapters.datasets.ml_preparation import prepare_dataset

prepared = prepare_dataset(DATASET, seed=0)
print("sizes:", {k: v.metadata["size"] for k, v in prepared.partitions.items()})
print("dist:", prepared.metadata["class_distribution"])
assert sum(v.metadata["size"] for v in prepared.partitions.values()) == len(prepared.enriched) == 297
assert not hasattr(prepared.preprocessor, "transformers_")


In [ ]:
from types import SimpleNamespace

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

SEED = 0
CLASS_ORDER = ("baja", "media", "alta")
NUMERICAS = [c for c in prepared.X.columns if c != "tipo_zona"]


# Preprocesador con escala en numericas (logistica/knn/svc lo necesitan).
def _escalado():
    return ColumnTransformer(
        [
            ("zona", OneHotEncoder(handle_unknown="ignore"), ["tipo_zona"]),
            ("numericas", StandardScaler(), NUMERICAS),
        ]
    )


MODELOS = {
    "arbol": (
        lambda pre: make_pipeline(clone(pre), DecisionTreeClassifier(random_state=SEED)),
        {
            "decisiontreeclassifier__max_depth": [None, 3, 5],
            "decisiontreeclassifier__min_samples_split": [2, 5],
        },
    ),
    "logistica": (
        lambda pre: make_pipeline(_escalado(), LogisticRegression(max_iter=2000, random_state=SEED)),
        {"logisticregression__C": [0.5, 1.0, 2.0]},
    ),
    "knn": (
        lambda pre: make_pipeline(_escalado(), KNeighborsClassifier()),
        {"kneighborsclassifier__n_neighbors": [3, 5, 7]},
    ),
    "svc": (
        lambda pre: make_pipeline(_escalado(), SVC(kernel="rbf", random_state=SEED)),
        {"svc__C": [0.5, 1.0]},
    ),
}

train_idx = list(prepared.partitions["train"].indices)
val_idx = list(prepared.partitions["validation"].indices)
X_train, y_train = prepared.X.iloc[train_idx], prepared.y.iloc[train_idx]
X_val, y_val = prepared.X.iloc[val_idx], prepared.y.iloc[val_idx]
assert y_train.value_counts().min() >= 5, "el CV de 5 folds necesita 5 ejemplos por clase en train"

cv_por_dentro = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
metricas = []
for model_id, (constructor, grilla) in MODELOS.items():
    base = constructor(prepared.preprocessor)
    cv_res = cross_validate(clone(base), X_train, y_train, cv=cv_por_dentro, scoring=["accuracy", "f1_macro"])
    busqueda = GridSearchCV(
        clone(base),
        grilla,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
        scoring="f1_macro",
        refit=True,
    )
    busqueda.fit(X_train, y_train)
    pred_val = busqueda.predict(X_val)
    metricas.append(
        SimpleNamespace(
            model_id=model_id,
            cv_accuracy_mean=float(cv_res["test_accuracy"].mean()),
            cv_accuracy_std=float(cv_res["test_accuracy"].std()),
            cv_f1_macro_mean=float(cv_res["test_f1_macro"].mean()),
            cv_f1_macro_std=float(cv_res["test_f1_macro"].std()),
            best_params=dict(busqueda.best_params_),
            validation_accuracy=float(accuracy_score(y_val, pred_val)),
            validation_f1_macro=float(f1_score(y_val, pred_val, average="macro", zero_division=0)),
            validation_confusion_matrix=tuple(
                tuple(int(v) for v in fila)
                for fila in confusion_matrix(y_val, pred_val, labels=list(CLASS_ORDER)).tolist()
            ),
            test_accuracy=None,
            test_f1_macro=None,
        )
    )

comparison = SimpleNamespace(
    metrics=metricas,
    chosen_model_id=max(metricas, key=lambda m: (m.validation_f1_macro, m.validation_accuracy)).model_id,
)
for m in comparison.metrics:
    print(f"{m.model_id}: cv_f1={m.cv_f1_macro_mean:.4f} val_f1={m.validation_f1_macro:.4f} best={m.best_params}")
    print(f"  matriz_val={list(list(r) for r in m.validation_confusion_matrix)}")
print("elegido:", comparison.chosen_model_id)
assert all(m.test_accuracy is None for m in comparison.metrics)
mejor = max(comparison.metrics, key=lambda m: (m.validation_f1_macro, m.validation_accuracy))
assert comparison.chosen_model_id == mejor.model_id


In [ ]:
# Test un solo uso, solo para el elegido: re-ajuste en train y una sola prediccion en test.
elegido = comparison.chosen_model_id
constructor, _ = MODELOS[elegido]
mejores_params = next(m.best_params for m in comparison.metrics if m.model_id == elegido)
test_idx = list(prepared.partitions["test"].indices)
X_test, y_test = prepared.X.iloc[test_idx], prepared.y.iloc[test_idx]
tuberia_final = constructor(prepared.preprocessor)
tuberia_final.set_params(**mejores_params)
tuberia_final.fit(X_train, y_train)
pred_test = tuberia_final.predict(X_test)

for m in comparison.metrics:
    if m.model_id == elegido:
        m.test_accuracy = float(accuracy_score(y_test, pred_test))
        m.test_f1_macro = float(f1_score(y_test, pred_test, average="macro", zero_division=0))
final = SimpleNamespace(metrics=comparison.metrics, chosen_model_id=elegido)
elegida = next(m for m in final.metrics if m.model_id == final.chosen_model_id)
print(f"test un solo uso ({elegida.model_id}): acc={elegida.test_accuracy:.4f} f1={elegida.test_f1_macro:.4f}")
assert elegida.test_accuracy is not None and elegida.test_f1_macro is not None
assert sum(1 for m in final.metrics if m.test_accuracy is not None) == 1
print("NOTA: metricas = ajuste a la regla sintetica, no desempeno operativo.")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.tree import plot_tree
from IPython.display import display
from collection.adapters.models.model_specs import build_pipeline

ROOT = next(r for r in ROOTS if (r / "dataset" / "salida" / "tramos.csv").is_file())
FIGURES_DIR = ROOT / "docs" / "figs"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
CLASS_ORDER = ("baja", "media", "alta")

# CV F1 macro (barras de error = std) frente a F1 macro de validación.
fig, ax = plt.subplots(figsize=(8, 5))
models = comparison.metrics
labels = [metric.model_id for metric in models]
positions = np.arange(len(models))
ax.bar(positions - 0.2, [m.cv_f1_macro_mean for m in models], width=0.4, yerr=[m.cv_f1_macro_std for m in models], capsize=4, label="CV F1 macro ± std")
ax.bar(positions + 0.2, [m.validation_f1_macro for m in models], width=0.4, label="Validación F1 macro")
ax.set(xticks=positions, xticklabels=labels, ylim=(0, 1.05), ylabel="F1 macro", title="CV (solo train) vs validación")
ax.legend(); fig.tight_layout()
path = FIGURES_DIR / "f1_comparacion.png"
fig.savefig(path, dpi=150); display(fig); plt.close(fig); print(path)

# Matrices de validación en orden baja/media/alta.
fig, axes = plt.subplots(1, len(models), figsize=(4 * len(models), 4))
for ax, metric in zip(axes, models, strict=True):
    ConfusionMatrixDisplay(confusion_matrix=np.asarray(metric.validation_confusion_matrix), display_labels=CLASS_ORDER).plot(ax=ax, colorbar=False, cmap="Blues", values_format="d")
    ax.set_title(metric.model_id)
fig.tight_layout()
path = FIGURES_DIR / "matrices_confusion_validacion.png"
fig.savefig(path, dpi=150); display(fig); plt.close(fig); print(path)

# Árbol ganador re-ajustado únicamente en train, mostrado hasta profundidad 3.
train_indices = prepared.partitions["train"].indices
tree = build_pipeline("arbol", prepared.preprocessor, seed=0)
tree.fit(prepared.X.iloc[list(train_indices)], prepared.y.iloc[list(train_indices)])
preprocessor_name = next(name for name in tree.named_steps if name != "decisiontreeclassifier")
fig, ax = plt.subplots(figsize=(18, 9))
plot_tree(tree.named_steps["decisiontreeclassifier"], feature_names=tree.named_steps[preprocessor_name].get_feature_names_out(), class_names=list(CLASS_ORDER), filled=True, rounded=True, max_depth=3, ax=ax)
ax.set_title("Árbol ganador (ajustado solo en train, profundidad máxima 3)")
fig.tight_layout()
path = FIGURES_DIR / "arbol_ganador.png"
fig.savefig(path, dpi=150); display(fig); plt.close(fig); print(path)


## Registro de elección: `arbol`
La comparación de arriba es el proceso de elección del modelo del proyecto; el backend no compara, solo sirve el árbol ya elegido.
- **Criterio**: mayor F1 macro en validación (desempate: accuracy). El test se usa una sola vez al final, solo para el elegido.
- **Por qué gana el árbol**: la etiqueta es sintética (score = 2*días + contenedores + zona + 2*quejas) y el árbol puede representar esa regla tal cual.
- **Límite**: las métricas miden ajuste a la regla construida, no desempeño operativo real.
- **Evidencia**: tablas impresas en este notebook + `docs/figs/f1_comparacion.png` y `docs/figs/matrices_confusion_validacion.png`.
- **Artefacto**: `backend/artifacts/arbol-montevideo.joblib`, generado con `scripts/generate_arbol_artifact.py` (re-ajuste en train con los mejores hiperparámetros del contraste en validación).